In [ ]:
from visualization_utils import TruthData
from utils import available_activation_datasets, available_activation_layers
import torch
from plotly.subplots import make_subplots
import configparser

model = 'llama-2-7b'
config = configparser.ConfigParser()
config.read('config.ini')
noperiod = config.getboolean(model, 'noperiod')

datasets_with_acts = available_activation_datasets(model, noperiod=noperiod)
if not datasets_with_acts:
    raise ValueError(f"No activations found for {model}. Run generate_acts.py first.")

common_layers = sorted(set.intersection(*[
    set(available_activation_layers(dataset, model, noperiod=noperiod))
    for dataset in datasets_with_acts
]))
if not common_layers:
    raise ValueError(f"No shared activation layers found for {model}: {datasets_with_acts}")

configured_layer = config.getint(model, 'probe_layer')
layer = configured_layer if configured_layer in common_layers else common_layers[-1]

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Using model={model}, layer={layer}, datasets={datasets_with_acts}, device={device}")

In [ ]:
TruthData.from_datasets(
    ['cities', 'neg_cities'], # datasets to use
    model=model,
    layer=layer,
    center=True,
    noperiod=noperiod,
    device=device
).plot(
    dimensions=2, # 3 dimensions also supported
    dim_offset=0, # increase if you want to ignore the first few PCs
    color='label',
    # if you don't want to plot and do PCA on all datasets:
    # plot_datasets = [ list of datasets to plot here ],
    # pca_datasets = [ list of datasets to use for PCA here]
)

In [ ]:
# visualization of all datasets with saved activations
import math

datasets = datasets_with_acts
cols = min(4, len(datasets))
rows = math.ceil(len(datasets) / cols)

td = TruthData.from_datasets(
    datasets,
    model=model,
    layer=layer,
    center=True,
    noperiod=noperiod,
    device=device)

fig = make_subplots(rows=rows, cols=cols, subplot_titles=datasets, horizontal_spacing=0.05, vertical_spacing=0.05)
for i, dataset in enumerate(datasets):
    for data in td.plot(
        dimensions=2,
        color='label',
        plot_datasets = [dataset],
        pca_datasets = [dataset],
        ).data:
        fig.add_trace(
            data, row=(i // cols) + 1, col=(i % cols) + 1
        )

for row in range(rows):
    for col in range(cols):
        fig.update_yaxes(
            scaleanchor = f"x{row * cols + col + 1}",
            scaleratio = 1,
            row=row+1,
            col=col+1,
        )

fig.update_coloraxes(
    colorscale = 'Bluered_r'
)

fig.update_layout(
    height=350 * rows,
    width=300 * cols,
    showlegend=False,
    title = {
        'text' : 'PCA visualizations of available datasets',
        'font' : {'size' : 30}
    },
    coloraxis_showscale=False,
)
fig.show()

In [ ]:
pca_datasets = datasets_with_acts
plot_datasets = datasets_with_acts

fig = make_subplots(rows=len(pca_datasets), cols=len(plot_datasets), 
    vertical_spacing=0.05, horizontal_spacing=0.01,
    )

for col, plot_dataset in enumerate(plot_datasets):
    fig.update_xaxes(title= {
                    'text': plot_dataset,
                    'font' : {'size' : 20}
                    },
                    row=len(pca_datasets), col=col+1
                )
for row, pca_dataset in enumerate(pca_datasets):
    fig.update_yaxes(title= {
                    'text': pca_dataset,
                    'font' : {'size' : 20}
                    },
                    row=row+1, col=1
                )

td = TruthData.from_datasets(plot_datasets, model=model, layer=layer, center=True, noperiod=noperiod, device=device)

for row, pca_dataset in enumerate(pca_datasets):
    for col, plot_dataset in enumerate(plot_datasets):
        subfig = td.plot(
            dimensions = 2,
            plot_datasets = [plot_dataset],
            pca_datasets = [pca_dataset],
            color='label',
        )
        fig.add_trace(subfig.data[0], row=row+1, col=col+1)

for row in range(len(pca_datasets)):
    for col in range(len(plot_datasets)):
        fig.update_yaxes(
            scaleanchor = f"x{row * len(plot_datasets) + col + 1}",
            scaleratio = 1,
            row=row+1,
            col=col+1,
        )

fig.update_coloraxes(
    colorscale = 'Bluered_r'
)

fig.update_layout(
    height=400 * len(pca_datasets),
    width=450 * len(plot_datasets),
    coloraxis_showscale=False,
    title = {
        'text': f'Dataset visualizations in available PCA bases',
        'font' : {'size' : 30}
    }
)

fig.show()

In [ ]:
from plotly.subplots import make_subplots

candidate_pairs = [
    ['cities', 'neg_cities'],
    ['sp_en_trans', 'neg_sp_en_trans'],
    ['larger_than', 'smaller_than'],
]
pairs = [pair for pair in candidate_pairs if all(dataset in datasets_with_acts for dataset in pair)]

# colormappings under the Rainbow colorscale
RED = 1
BLUE = .17
PURPLE = 0
YELLOW = .73

fig = make_subplots(rows=1, cols=len(pairs),
                    shared_yaxes=True,
                    x_title='PC1', y_title='PC2',
                    subplot_titles=[ '+'.join([dataset for dataset in pair]) for pair in pairs]
                    )

for i, pair in enumerate(pairs):
    td = TruthData.from_datasets(
        pair,
        model=model,
        layer=layer,
        noperiod=noperiod,
        device=device
    )

    td.df.loc[pair[0], 'label'] = td.df.loc[pair[0]]['label'].apply(lambda x: BLUE if x == 1 else RED)
    td.df.loc[pair[1], 'label'] = td.df.loc[pair[1]]['label'].apply(lambda x: YELLOW if x == 1 else PURPLE)

    subfig = td.plot(
        dimensions=2,
        color = 'label',
    )

    fig.add_trace(subfig.data[0], row=1, col=i+1)

fig.update_coloraxes(
    colorscale='Rainbow'
)

fig.show()

In [ ]:
from plotly.subplots import make_subplots

datasets = datasets_with_acts
layers = common_layers

figs = [[] for _ in datasets]

for i, dataset in enumerate(datasets):
    for sweep_layer in layers:
        fig = TruthData.from_datasets(
            [dataset],
            model=model,
            layer=sweep_layer,
            noperiod=noperiod,
            device=device
            ).plot(
                dimensions=2,
                color='label',
            )
        figs[i].append(fig)

fig = make_subplots(rows = len(datasets), cols = len(layers),
                    subplot_titles=[f"layer {sweep_layer}" for sweep_layer in layers],
                    vertical_spacing=0.05)

for i, dataset in enumerate(datasets):
    for j, sweep_layer in enumerate(layers):
        for data in figs[i][j].data:
            data['showlegend'] = False
            fig.add_trace(data, row=i+1, col=j+1)

for i, dataset in enumerate(datasets):
    fig.update_yaxes(title_text=dataset, row=i+1, col=1)

fig.update_coloraxes(
    colorscale='Bluered_r'
)

fig.update_layout(height=350 * len(datasets), width=300 * len(layers), coloraxis_showscale=False)

fig.update_layout(
    title = {
        'text' : f"Dataset visualizations across available layers",
        'font' : {'size' : 20},
    }
)

fig.show()